In [1]:
%pip install -q qdrant-client rank_bm25 sentence-transformers pandas numpy

import sys
from pathlib import Path

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "src").exists()), here)
sys.path.insert(0, str(ROOT / "src"))
import chatbot as cb

MODEL_DIR = cb.locate(ROOT, "kinya-retriever-e5", True) or cb.locate(ROOT, "kinya-retriever", True)
PASSAGES  = cb.locate(ROOT, "passages.jsonl")
QUERIES   = cb.locate(ROOT, "queries.jsonl")
PAIRS_DEV = cb.locate(ROOT, "pairs_dev.csv")

print("ROOT      :", ROOT)
print("MODEL_DIR :", MODEL_DIR)
print("PASSAGES  :", PASSAGES)
print("QUERIES   :", QUERIES)
print("PAIRS_DEV :", PAIRS_DEV)


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\PC\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
ROOT      : C:\Users\PC\Desktop\Mfasha chatbot
MODEL_DIR : None
PASSAGES  : C:\Users\PC\Desktop\Mfasha chatbot\data\passages.jsonl
QUERIES   : C:\Users\PC\Desktop\Mfasha chatbot\data\queries.jsonl
PAIRS_DEV : C:\Users\PC\Desktop\Mfasha chatbot\data\pairs_dev.csv


In [2]:
import os

skip = {".venv", "venv", ".git", "__pycache__", ".ipynb_checkpoints", "node_modules"}

print("Top-level items in ROOT:", sorted(p.name for p in ROOT.iterdir()))
print()
found = False
for cur, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if d not in skip]
    if "model.safetensors" in files:
        size = os.path.getsize(os.path.join(cur, "model.safetensors")) / 1e9
        print(f"model.safetensors: {size:.2f} GB  in  {cur}")
        found = True
    if "kinya" in os.path.basename(cur).lower():
        print("folder with 'kinya' in its name:", cur)
if not found:
    print("No model.safetensors found anywhere under ROOT")

Top-level items in ROOT: ['.env', '.venv', 'data', 'models', 'notebook', 'requirements.txt', 'results', 'src']

model.safetensors: 1.11 GB  in  C:\Users\PC\Desktop\Mfasha chatbot\models


In [3]:
MODEL_DIR = ROOT / "models"

need = ["model.safetensors", "config.json", "modules.json",
        "config_sentence_transformers.json", "sentence_bert_config.json",
        "tokenizer.json", "tokenizer_config.json"]

for n in need:
    print("OK      " if (MODEL_DIR / n).exists() else "MISSING ", n)

pool = MODEL_DIR / "1_Pooling"
print("OK      " if (pool / "config.json").exists() else "MISSING ", "1_Pooling/config.json")

others = sorted(p.name for p in MODEL_DIR.iterdir()
                if p.name not in need and p.name != "1_Pooling")
print("\nOther items in models:", others)

OK       model.safetensors
OK       config.json
OK       modules.json
OK       config_sentence_transformers.json
OK       sentence_bert_config.json
OK       tokenizer.json
OK       tokenizer_config.json
OK       1_Pooling/config.json

Other items in models: []


In [4]:
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(str(MODEL_DIR), device=DEVICE)
model.max_seq_length = 256

passages = pd.read_json(open(PASSAGES, encoding="utf-8"), lines=True, dtype=False).astype(str)
queries  = pd.read_json(open(QUERIES,  encoding="utf-8"), lines=True, dtype=False).astype(str)
qid2text = dict(zip(queries.qid, queries.text))
passage_texts = passages.text.tolist()
pid2idx = {p: i for i, p in enumerate(passages.pid)}

pairs_dev = pd.read_csv(PAIRS_DEV, dtype=str)
dev_q = pairs_dev.qid.map(qid2text).tolist()
gold  = np.array([pid2idx[p] for p in pairs_dev.pos_pid])

print("device:", DEVICE)
print("passages:", len(passage_texts), "| dev questions:", len(dev_q))
v = model.encode(["soya ihunikwa ahantu hameze gute ?"], normalize_embeddings=True)
print("embedding size:", v.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

device: cpu
passages: 984 | dev questions: 196
embedding size: (1, 768)


In [5]:
tk = model.tokenizer
tok_lens  = np.array([len(tk.encode(t)) for t in passage_texts])
char_lens = np.array([len(t) for t in passage_texts])
cpt = char_lens.sum() / tok_lens.sum()

print("passages over 256 tokens:", int((tok_lens > 256).sum()), "of", len(tok_lens))
print("passages over 200 tokens:", int((tok_lens > 200).sum()))
print("median / max tokens     :", int(np.median(tok_lens)), "/", int(tok_lens.max()))
print("chars per token: %.2f  ->  200 tokens is about %d characters" % (cpt, 200 * cpt))

Token indices sequence length is longer than the specified maximum sequence length for this model (498 > 256). Running this sequence through the model will result in indexing errors


passages over 256 tokens: 306 of 984
passages over 200 tokens: 435
median / max tokens     : 176 / 1086
chars per token: 2.67  ->  200 tokens is about 534 characters


In [6]:
import re

MAX_TOKENS = 200
MAX_CHARS  = int(MAX_TOKENS * cpt)

def hard_split(sentence, max_chars):
    out, cur = [], ""
    for w in sentence.split():
        if cur and len(cur) + 1 + len(w) > max_chars:
            out.append(cur); cur = w
        else:
            cur = (cur + " " + w).strip()
    if cur: out.append(cur)
    return out

def split_passage(text, max_chars=MAX_CHARS):
    if len(text) <= max_chars:
        return [text]                                    # short passages stay whole
    sents = []
    for s in re.split(r"(?<=[.;?!])\s+", text):
        sents += hard_split(s, max_chars) if len(s) > max_chars else [s]
    chunks, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur + [s])) > max_chars:
            chunks.append(" ".join(cur))
            cur = cur[-1:] if len(cur[-1]) < max_chars / 3 else []   # small overlap
        cur.append(s)
    if cur: chunks.append(" ".join(cur))
    return chunks

chunks = []
for pidx, (pid, text) in enumerate(zip(passages.pid, passage_texts)):
    for n, c in enumerate(split_passage(text)):
        chunks.append({"pid": pid, "pidx": pidx, "chunk_no": n, "text": c})

chunk_texts = [c["text"] for c in chunks]
chunk2p = np.array([c["pidx"] for c in chunks])
n_split = len(set(c["pidx"] for c in chunks if c["chunk_no"] > 0))
chunk_tokens = np.array([len(tk.encode(t)) for t in chunk_texts])

print(f"{len(passage_texts)} passages -> {len(chunks)} chunks | passages that were split: {n_split}")
print("chunks over 256 tokens:", int((chunk_tokens > 256).sum()), "| longest chunk:", int(chunk_tokens.max()), "tokens")

984 passages -> 1705 chunks | passages that were split: 439
chunks over 256 tokens: 0 | longest chunk: 247 tokens


In [7]:
import json
from pathlib import Path

DATA_DIR = Path(PASSAGES).parent

P = model.encode(passage_texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
C = model.encode(chunk_texts,   batch_size=32, normalize_embeddings=True, show_progress_bar=True)
Q = model.encode(dev_q,         batch_size=32, normalize_embeddings=True, show_progress_bar=True)

with open(DATA_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
np.save(DATA_DIR / "chunk_emb_e5.npy", C)

print("shapes  passages:", P.shape, "| chunks:", C.shape, "| dev questions:", Q.shape)
print("saved to:", DATA_DIR)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   0%|          | 0/54 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

shapes  passages: (984, 768) | chunks: (1705, 768) | dev questions: (196, 768)
saved to: C:\Users\PC\Desktop\Mfasha chatbot\data


In [8]:
from rank_bm25 import BM25Okapi

tok = cb.tok

def mm(S):                                   # row-wise min-max scaling
    mn, mx = S.min(1, keepdims=True), S.max(1, keepdims=True)
    return (S - mn) / (mx - mn + 1e-9)

def evaluate(S, name):
    g = S[np.arange(len(S)), gold][:, None]
    r = (S >= g).sum(1)
    return {"method": name,
            "Recall@1": (r <= 1).mean(), "Recall@5": (r <= 5).mean(),
            "Recall@10": (r <= 10).mean(),
            "MRR@10": np.where(r <= 10, 1 / r, 0).mean()}

def to_passage(S):                           # best chunk represents its passage
    out = np.full((S.shape[0], len(passage_texts)), -1e9)
    for p in np.unique(chunk2p):
        out[:, p] = S[:, chunk2p == p].max(1)
    return out

bm25_p = BM25Okapi([tok(t) for t in passage_texts])
bm25_c = BM25Okapi([tok(t) for t in chunk_texts])

S_emb_p = Q @ P.T
S_bm_p  = np.vstack([bm25_p.get_scores(tok(q)) for q in dev_q])
S_emb_c = Q @ C.T
S_bm_c  = np.vstack([bm25_c.get_scores(tok(q)) for q in dev_q])

rows = [
    evaluate(S_bm_p,                                  "BM25, whole passages"),
    evaluate(S_emb_p,                                 "Fine-tuned, whole passages"),
    evaluate(0.8 * mm(S_emb_p) + 0.2 * mm(S_bm_p),    "Hybrid 0.8, whole passages"),
    evaluate(to_passage(S_bm_c),                      "BM25, chunks"),
    evaluate(to_passage(S_emb_c),                     "Fine-tuned, chunks"),
    evaluate(to_passage(0.8 * mm(S_emb_c) + 0.2 * mm(S_bm_c)), "Hybrid 0.8, chunks"),
]
cmp = pd.DataFrame(rows).set_index("method")
print(cmp.round(3).to_string())
cmp.round(4).to_csv(ROOT / "results" / "chunking_comparison_dev.csv")

                            Recall@1  Recall@5  Recall@10  MRR@10
method                                                           
BM25, whole passages           0.520     0.776      0.857   0.627
Fine-tuned, whole passages     0.735     0.918      0.939   0.809
Hybrid 0.8, whole passages     0.760     0.913      0.949   0.829
BM25, chunks                   0.505     0.776      0.837   0.619
Fine-tuned, chunks             0.724     0.913      0.934   0.799
Hybrid 0.8, chunks             0.770     0.898      0.949   0.830


In [9]:
import time

bot = cb.Chatbot(chunks_path=DATA_DIR / "chunks.jsonl",
                 emb_path=DATA_DIR / "chunk_emb_e5.npy",
                 model=model)
print("chunks in database:", len(bot.chunks))

t0 = time.time()
h1 = h5 = 0
for q, g in zip(dev_q, pairs_dev.pos_pid):
    res, _ = bot.search(q, k=5)
    pids = [r["pid"] for r in res]
    h1 += (pids[0] == g)
    h5 += (g in pids)

print(f"Qdrant hybrid on dev  Recall@1: {h1/len(dev_q):.3f} | Recall@5: {h5/len(dev_q):.3f}")
print(f"time: {time.time()-t0:.0f} s for {len(dev_q)} questions ({(time.time()-t0)/len(dev_q):.2f} s each)")

chunks in database: 1705
Qdrant hybrid on dev  Recall@1: 0.745 | Recall@5: 0.908
time: 53 s for 196 questions (0.27 s each)


In [10]:
off_topic = [
    "Ni gute nakora porogaramu ya telefoni?",
    "Ni nde wabaye perezida wa Amerika mu 2020?",
    "Ni gute nakwita ku mubiri wanjye ngo nkomere mu myitozo ya siporo?",
    "Ikibazo cy'ubukungu bw'u Buyapani ni ikihe?",
    "Ni gute nateka pizza?",
    "Ni iki cyiza cyo kureba kuri televiziyo uyu munsi?",
    "Ni gute nabona akazi muri banki?",
    "Ikipe y'umupira w'amaguru ya Arsenal yatsinze ryari?",
]

dev_conf = (Q @ C.T).max(1)
off_vecs = model.encode(off_topic, normalize_embeddings=True)
off_conf = (off_vecs @ C.T).max(1)

print("In-scope  : min %.3f | 5th pct %.3f | median %.3f" %
      (dev_conf.min(), np.percentile(dev_conf, 5), np.median(dev_conf)))
print("Off-topic :", np.round(np.sort(off_conf), 3))

rows = []
for t in (0.40, 0.45, 0.485, 0.50, 0.55, 0.60):
    rows.append({"threshold": t,
                 "real questions answered %": round(100 * (dev_conf >= t).mean(), 1),
                 "off-topic refused (of 8)": int((off_conf < t).sum())})
print(pd.DataFrame(rows).to_string(index=False))

In-scope  : min 0.406 | 5th pct 0.505 | median 0.768
Off-topic : [0.333 0.408 0.415 0.452 0.454 0.464 0.552 0.563]
 threshold  real questions answered %  off-topic refused (of 8)
     0.400                      100.0                         1
     0.450                       98.5                         3
     0.485                       96.4                         6
     0.500                       95.4                         6
     0.550                       85.7                         6
     0.600                       79.6                         8


In [12]:
bot.threshold = 0.485

tests = dev_q[:3] + off_topic[:3]
for q in tests:
    r = bot.ask(q)
    print(f"Q: {q}")
    print(f"   refused: {r['refused']} | confidence: {r['confidence']:.3f} | passage: {r['passage_id']}")
    print(f"   ANSWER (best chunk): {r['answer'][:400]}")
    if not r["refused"]:
        print(f"   SHORT (2 best pieces): {r['short'][:300]}")

# does the soya passage mention equipment at all?
pid = bot.ask(dev_q[1])["passage_id"]
print("\nAll chunks of passage", pid, "for the question:", dev_q[1])
for c in bot.by_pid[pid]:
    print(f"[chunk {c['chunk_no']}] {cb.clean(c['text'])}\n")

Q: GUHUNIKA SOYA .
   refused: False | confidence: 0.892 | passage: A424T233
   ANSWER (best chunk): Guhunika soya bikorwa mu buryo bukurikira: Soya ihunikwa igomba kuba yumye neza ku buryo igipimo cy'amazi mu ntete kitajya hejuru ya 11%. (Iyo udafite ibikoresho bipima ubu buhehere ukoresha iryinyo iyo soya ihise imeneka iba yumye neza), Ihunikwa ahantu hari umwuka, humutse kandi hatagera ibyonnyi. Imifuka igomba gupangwa ku dutanda tw'imbaho (palettes) ntigire aho ihurira n'ubukonje bwo hasi.
   SHORT (2 best pieces): Guhunika soya bikorwa mu buryo bukurikira: Soya ihunikwa igomba kuba yumye neza ku buryo igipimo cy'amazi mu ntete kitajya hejuru ya 11%. Soya ihunikwa igihe gito (ntirenza ibihembwe2 by'ihinga) kubera ko yangizwa n'ubushyuhe ndetse n'ubuhehere birengeje urugero.
Q: mu guhunika soya bitondera iki ? bakenera ibihe bikoresho ?
   refused: False | confidence: 0.744 | passage: A424T233
   ANSWER (best chunk): Guhunika soya bikorwa mu buryo bukurikira: Soya ihunikwa igomba ku

In [13]:
import random
random.seed(7)

# 25 dev questions spread over the different dev passages
by_p = pd.DataFrame({"i": range(len(gold)), "pid": pairs_dev.pos_pid.values}).groupby("pid")["i"].apply(list)
pools = [lst[:] for lst in by_p]
for lst in pools: random.shuffle(lst)
picked = []
while len(picked) < 25 and any(pools):
    for lst in pools:
        if lst and len(picked) < 25:
            picked.append(lst.pop())

my_questions = []   # optional: add your own farmer-style questions in Kinyarwanda here

rows = []
for i in picked:
    r = bot.ask(dev_q[i], short=False)
    rows.append({"type": "dev", "question": dev_q[i], "expected_pid": pairs_dev.pos_pid.iloc[i],
                 "returned_pid": r["passage_id"], "confidence": round(r["confidence"], 3),
                 "refused": r["refused"],
                 "top1_correct": (r["passage_id"] == pairs_dev.pos_pid.iloc[i]),
                 "answer": r["answer"]})
for q in off_topic:
    r = bot.ask(q, short=False)
    rows.append({"type": "off-topic", "question": q, "expected_pid": "REFUSE",
                 "returned_pid": r["passage_id"], "confidence": round(r["confidence"], 3),
                 "refused": r["refused"], "top1_correct": r["refused"], "answer": r["answer"]})
for q in my_questions:
    r = bot.ask(q, short=False)
    rows.append({"type": "mine", "question": q, "expected_pid": "", "returned_pid": r["passage_id"],
                 "confidence": round(r["confidence"], 3), "refused": r["refused"],
                 "top1_correct": "", "answer": r["answer"]})

rev = pd.DataFrame(rows)
rev["helpful_yes_partly_no"] = ""     
rev["notes"] = ""
rev.to_csv(ROOT / "results" / "chatbot_review.csv", index=False, encoding="utf-8-sig")

d = rev[rev.type == "dev"]; o = rev[rev.type == "off-topic"]
ans = d[~d.refused]
print(f"dev questions answered       : {len(ans)} of {len(d)}")
print(f"right passage when answered  : {int(ans.top1_correct.sum())} of {len(ans)}")
print(f"off-topic refused            : {int(o.refused.sum())} of {len(o)}")
print("\nWrong or refused dev questions:")
print(d[~d.top1_correct][["question", "confidence", "refused"]].to_string(index=False))

dev questions answered       : 25 of 25
right passage when answered  : 19 of 25
off-topic refused            : 6 of 8

Wrong or refused dev questions:
                                                                                               question  confidence  refused
ni gute umuworozi amenya umubare w' inkoko ukwiye kwororerwa ku buso runaka hagendewe ku minsi zimaze ?       0.534    False
                                               Ni ibiki ukwiye gusuzuma mu gihe ugiye kugura kurujete ?       0.650    False
                                                             nii ikihe gihe kiza cyo gutera imyumbati ?       0.670    False
                       nigute umworozi yamenya umubare w' imishwi agura agendeye kuko ikiraro kingana ?       0.569    False
                                                                      uburima burangwa niki muri pome ?       0.582    False
                                                        Ni iyihe ngano y' inkoko n' ubusobzororerwa

In [14]:
passage_by_pid = {str(p): t for p, t in zip(passages.pid, passages.text)}
wrong = d[~d.top1_correct]

for _, row in wrong.iterrows():
    res, _ = bot.search(row.question, k=5)
    pids = [x["pid"] for x in res]
    rank = pids.index(row.expected_pid) + 1 if row.expected_pid in pids else None
    print("Q:", row.question)
    print(f"   confidence {row.confidence} | expected passage is at rank: {rank}")
    print("   EXPECTED :", cb.clean(passage_by_pid.get(row.expected_pid, ""))[:300])
    print("   RETURNED :", cb.clean(passage_by_pid.get(row.returned_pid, ""))[:300])
ok, bad = d[d.top1_correct].confidence, d[~d.top1_correct].confidence
print(f"confidence when right: mean {ok.mean():.3f} | when wrong: mean {bad.mean():.3f}")

Q: ni gute umuworozi amenya umubare w' inkoko ukwiye kwororerwa ku buso runaka hagendewe ku minsi zimaze ?
   confidence 0.534 | expected passage is at rank: 2
   EXPECTED : Umubare w'inkoko n'ubuso zakororerwaho, hagendewe kuminsi imishwi imaze: Iminsi 1 - 10: Imishwi 40 / kuri meterio kare 1; Iminsi 11 - 20: Imishwi 30 / Kuri metero kare 1; Iminsi 21 - 30:
   RETURNED : Gukingira inkoko z'inyama; Igihe inkoko zimaze iminsi 2: Inkoko zikorerwa urukingo rwa Umuraramo (New Castle). Igipimo cy'umuti ni New Castle HB 1: agacupa k'inkingo 1000 bashyira muri litiro 20 z'amazi meza. Igihe inkoko zimaze iminsi 1 kugeza kuri 4: Inkoko zihabwa Vitamine (Anti - stress). Igipim
Q: Ni ibiki ukwiye gusuzuma mu gihe ugiye kugura kurujete ?
   confidence 0.65 | expected passage is at rank: 2
   EXPECTED : Ahantu ho kugura umurama wa kurujete: Gura imbuto / inzuzi ahantu hizewe hagurirwa inyongeramusaruro kandi mbere yo kugura reba igihe ubushobozi bwo kumera buzarangirira. Kuri ari 1 hakenerwa garama

In [15]:
rows = []
for q, g in zip(dev_q, pairs_dev.pos_pid):
    res, conf = bot.search(q, k=5)
    pids = [r["pid"] for r in res]
    rows.append({"conf": conf, "rank": pids.index(g) + 1 if g in pids else 99})
ev = pd.DataFrame(rows)

print("Recall@1: %.3f | Recall@3: %.3f | Recall@5: %.3f" %
      ((ev["rank"] <= 1).mean(), (ev["rank"] <= 3).mean(), (ev["rank"] <= 5).mean()))
print()

bands = [(0.0, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 2.0)]
out = []
for lo, hi in bands:
    sub = ev[(ev.conf >= lo) & (ev.conf < hi)]
    if len(sub):
        out.append({"confidence": f"{lo:.1f} to {min(hi,1):.1f}", "questions": len(sub),
                    "best answer correct %": round(100 * (sub["rank"] <= 1).mean(), 1),
                    "correct in top 3 %": round(100 * (sub["rank"] <= 3).mean(), 1)})
print(pd.DataFrame(out).to_string(index=False))

Recall@1: 0.745 | Recall@3: 0.867 | Recall@5: 0.908

confidence  questions  best answer correct %  correct in top 3 %
0.0 to 0.6         40                   30.0                57.5
0.6 to 0.7         35                   57.1                85.7
0.7 to 0.8         47                   87.2                93.6
0.8 to 1.0         74                   98.6                98.6
